# SpaceX Falcon 9 Machine Learning Prediction

This notebook uses machine learning classification models to predict Falcon 9 first-stage landing success.

The workflow includes feature preparation, train-test splitting, model training, evaluation, and comparison of classification models.

## 1. Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load SpaceX Launch Data

In [ ]:
response = requests.get("https://api.spacexdata.com/v4/launches/past")
response.raise_for_status()

launches = response.json()
df = pd.json_normalize(launches)

rocket_response = requests.get("https://api.spacexdata.com/v4/rockets")
rocket_response.raise_for_status()

rockets = rocket_response.json()
rocket_lookup = {rocket["id"]: rocket["name"] for rocket in rockets}
    
    df["rocket_name"] = df["rocket"].map(rocket_lookup)
    df = df[df["rocket_name"].str.contains("Falcon 9", na=False)].copy()
    
    df["success"] = df["success"].astype("Int64")
    df["year"] = pd.to_datetime(df["date_utc"], errors="coerce").dt.year
    
    print("Falcon 9 records:", len(df))

## 3. Create the Target Variable

The target variable represents whether the launch was successful.

In [ ]:
model_df = df[["success", "year", "rocket_name"]].copy()
model_df = model_df.dropna(subset=["success", "year"])

model_df["success"] = model_df["success"].astype(int)
model_df["rocket_code"] = pd.Categorical(model_df["rocket_name"]).codes

model_df.head()

## 4. Prepare Features and Target

In [ ]:
X = model_df[["year", "rocket_code"]]
y = model_df["success"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 5. Standardize Features

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Logistic Regression

In [ ]:
logistic_model = LogisticRegression(random_state=42)
logistic_model.fit(X_train_scaled, y_train)

logistic_predictions = logistic_model.predict(X_test_scaled)
logistic_accuracy = accuracy_score(y_test, logistic_predictions)

print("Logistic Regression Accuracy:", logistic_accuracy)

## 7. Support Vector Machine

In [ ]:
svm_model = SVC(kernel="linear", random_state=42)
svm_model.fit(X_train_scaled, y_train)

svm_predictions = svm_model.predict(X_test_scaled)
svm_accuracy = accuracy_score(y_test, svm_predictions)

print("SVM Accuracy:", svm_accuracy)

## 8. Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print("Decision Tree Accuracy:", tree_accuracy)

## 9. K-Nearest Neighbors

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_scaled, y_train)

knn_predictions = knn_model.predict(X_test_scaled)
knn_accuracy = accuracy_score(y_test, knn_predictions)

print("KNN Accuracy:", knn_accuracy)

## 10. Compare Model Performance

In [ ]:
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Decision Tree",
        "KNN"
    ],
    "Accuracy": [
        logistic_accuracy,
        svm_accuracy,
        tree_accuracy,
        knn_accuracy
    ]
})

model_results = model_results.sort_values("Accuracy", ascending=False)
model_results

## 11. Select the Best Model

In [ ]:
best_model_name = model_results.iloc[0]["Model"]
best_accuracy = model_results.iloc[0]["Accuracy"]

print("Best Model:", best_model_name)
print("Best Accuracy:", best_accuracy)

## 12. Confusion Matrix

In [ ]:
best_predictions = {
    "Logistic Regression": logistic_predictions,
    "SVM": svm_predictions,
    "Decision Tree": tree_predictions,
    "KNN": knn_predictions
}[best_model_name]

cm = confusion_matrix(y_test, best_predictions)

print("Confusion Matrix:")
print(cm)

## 13. Classification Report

In [ ]:
print(classification_report(y_test, best_predictions, zero_division=0))

## Conclusion

Multiple classification models were trained and evaluated for Falcon 9 launch-success prediction. Model accuracy and the confusion matrix were used to compare performance and identify the strongest model.